# Comparison matching pipeline debug

This notebook mirrors `services.comparison_matcher.ComparisonMatcher` and the comparison service flow: load comparison, load extracted offers, build prompt, parse/complete LLM output, normalize matches, and optionally recalculate totals.

In [1]:
from pathlib import Path
import json
import sys

ROOT = Path.cwd()
if not (ROOT / 'services').exists():
    ROOT = ROOT.parent
sys.path.insert(0, str(ROOT))

from application.offer_service import OfferService
from matching.comparison_prompt import MATCH_RESPONSE_SCHEMA, build_comparison_match_prompt
from matching.match_normalizer import complete_response, normalize_matched_posts
from services.comparison_matcher import ComparisonMatcher, MATCHER_MODEL_ID
from services.extract_offer import ask_llm, parse_json_response
from services.folder_handler import FolderHandler
from services.project import Project

STORAGE_DIR = ROOT / 'storage'
PROJECT_NAME = '22.31_baksteen'
RUN_LLM = False
SAVE_RESULTS = False

folder_handler = FolderHandler(STORAGE_DIR)
project = Project(STORAGE_DIR / PROJECT_NAME, folder_handler)
matcher = ComparisonMatcher(folder_handler)
offer_service = OfferService(folder_handler)

print(f'Project: {project.path}')
print(f'Comparison exists: {project.comparison_path.exists()}')
print(f'Offers: {[offer.name for offer in project.offers()]}')

Project: /Users/timojolman/Zakelijk/UniPartners/vanWijnen/vanwijnen/development/app/storage/22.31_baksteen
Comparison exists: True
Offers: ['postma', 'zuidema']


## Load comparison and extracted offer results

In [2]:
comparison = project.load_comparison()
offer_results = matcher.project_offer_results(project)

print(f'Comparison posts: {len(comparison.get("Posten", []))}')
print(f'Existing matched posts: {len(comparison.get("MatchedPosten", []))}')
print(f'Offer results: {len(offer_results)}')
for offer in offer_results:
    print(f'- {offer["Bestand"]}: {len(offer.get("Posten", []))} extracted posts')

comparison.get('Posten', [])[:3]

Comparison posts: 8
Existing matched posts: 8
Offer results: 2
- postma.pdf: 23 extracted posts
- zuidema.pdf: 20 extracted posts


[{'Omschrijving': 'Vermetselen gevelsteen wildverband',
  'Aantal': '86,95',
  'Eenheid': 'dzd'},
 {'Omschrijving': 'toeslag wilverband', 'Aantal': '86,95', 'Eenheid': 'dzd'},
 {'Omschrijving': 'toeslag strooisteen', 'Aantal': '1112,00', 'Eenheid': 'm2'}]

In [3]:
offer_results[:2]

[{'Bestand': 'postma.pdf',
  'Posten': [{'Omschrijving': 'Gevelsteen B3 WF, basis halfsteens verband, vormbaksteen met normale vochtopname, incl. terugliggende geglazuurde gevelstenen. Start werk Q2 2026, start metselwerk Q3 2026',
    'Categorie': 'Metselwerk',
    'Totaalbedrag': 'ONBEKEND',
    'Eenheid': 'dzd',
    'Eenheidsprijs': '765.00',
    'Aantal': 'ONBEKEND'},
   {'Omschrijving': 'Vooropperen steen op en rondom de steiger met een door de aannemer ter beschikking gestelde spieringskraan.',
    'Categorie': 'Metselwerk',
    'Totaalbedrag': 'ONBEKEND',
    'Eenheid': 'ONBEKEND',
    'Eenheidsprijs': 'ONBEKEND',
    'Aantal': 'ONBEKEND'},
   {'Omschrijving': 'Alternatief inzet verreiker JCB 13.5m1 star t.b.v. vooropperen steen/ mortel op de steiger, incl. aan-afvoer.',
    'Categorie': 'Metselwerk',
    'Totaalbedrag': 'ONBEKEND',
    'Eenheid': 'dzd',
    'Eenheidsprijs': '70.00',
    'Aantal': 'ONBEKEND'},
   {'Omschrijving': 'Voegwerk, direct doorgestreken/ gepointerd',
   

## Build the exact matching prompt

In [4]:
prompt = build_comparison_match_prompt(comparison, offer_results)
print(f'Model: {MATCHER_MODEL_ID}')
print(f'Prompt characters: {len(prompt)}')
print(prompt[:4000])

Model: gemini-2.5-flash
Prompt characters: 13572
Je koppelt begrotings-/vergelijkingsregels aan offerteposten.

Vergelijkingsregels:
[
  {
    "Omschrijving": "Vermetselen gevelsteen wildverband",
    "Aantal": "86,95",
    "Eenheid": "dzd"
  },
  {
    "Omschrijving": "toeslag wilverband",
    "Aantal": "86,95",
    "Eenheid": "dzd"
  },
  {
    "Omschrijving": "toeslag strooisteen",
    "Aantal": "1112,00",
    "Eenheid": "m2"
  },
  {
    "Omschrijving": "inzet verreiker JCB 13.5m1 star t.b.v. vooropperen steen/mortel op de steiger, incl. aan-afvoer.",
    "Aantal": "86,95",
    "Eenheid": "dzd"
  },
  {
    "Omschrijving": "toeslag gebogen metselwerk",
    "Aantal": "178",
    "Eenheid": "m2"
  },
  {
    "Omschrijving": "vermetselen accentsteen 20mm verdiept",
    "Aantal": "3716,00",
    "Eenheid": "dzd"
  },
  {
    "Omschrijving": "steen zagen, wildverband,  0,5 mu/dzd",
    "Aantal": "86,95",
    "Eenheid": "dzd"
  },
  {
    "Omschrijving": "voegwerk doorstrijken",
    "Aanta

## Replay cached LLM response

This is the same parse and completion step used by `ComparisonMatcher.match_comparison_posts`, but it reads `comparison_llm_response.txt` instead of calling Gemini.

In [ ]:
cached_response_path = folder_handler.project_comparison_llm_response_path(project.path)
print(f'Cached response: {cached_response_path}')
print(f'Exists: {cached_response_path.exists()}')

if cached_response_path.exists():
    raw_answer = cached_response_path.read_text()
    parsed_match = parse_json_response(raw_answer)
    completed_match = complete_response(parsed_match, offer_results)
    print(f'Raw matched rows: {len(parsed_match.get("MatchedPosten", []))}')
    print(f'Completed matched rows: {len(completed_match.get("MatchedPosten", []))}')
else:
    raw_answer = None
    parsed_match = None
    completed_match = None
    print('No cached matching response found.')

In [ ]:
if completed_match:
    normalized_rows = normalize_matched_posts(comparison, completed_match, offer_results)
    print(f'Normalized rows: {len(normalized_rows)}')
    normalized_rows[:2]
else:
    normalized_rows = []
    print('Nothing to normalize yet.')

## Recalculate existing comparison totals

This mirrors `ComparisonMatcher.recalculate_matched_posts`, which refreshes matches from current `extract.json` files and recalculates totals.

In [ ]:
comparison_copy = json.loads(json.dumps(comparison))
recalculated_rows = matcher.recalculate_matched_posts(comparison_copy, project)
print(f'Recalculated rows: {len(recalculated_rows)}')
recalculated_rows[:2]

## Optional live matching

Set `RUN_LLM = True` in the setup cell to call Gemini. Set `SAVE_RESULTS = True` only when you want to overwrite `comparison.json` and `comparison_llm_response.txt` for the selected project.

In [ ]:
if RUN_LLM:
    live_answer = ask_llm(
        prompt,
        response_schema=MATCH_RESPONSE_SCHEMA,
        model=MATCHER_MODEL_ID,
        label='comparison_match_debug',
    )
    live_parsed = parse_json_response(live_answer)
    live_completed = complete_response(live_parsed, offer_results)
    live_normalized_rows = normalize_matched_posts(comparison, live_completed, offer_results)
    print(f'Live normalized rows: {len(live_normalized_rows)}')

    if SAVE_RESULTS:
        comparison_to_save = json.loads(json.dumps(comparison))
        comparison_to_save['MatchedPosten'] = live_normalized_rows
        folder_handler.save_comparison_llm_response(project, live_answer)
        project.save_comparison(comparison_to_save)
        print(f'Saved comparison to {project.comparison_path}')
else:
    print('RUN_LLM is False; skipped live matching.')